In [3]:
from TradingviewData import TradingViewData,Interval

request = TradingViewData()

In [4]:
request.search('AAVE','USD')

Expecting value: line 1 column 1 (char 0)


[]

In [6]:
import pandas as pd
import os

# ======================
# CONFIGURACIÓN GENERAL
# ======================
SYMBOL = "AAVEUSD"
EXCHANGE = "BINANCE"
YEARS = ["2022", "2023", "2024", "2025"]

BASE_PATH = "Data"

# ======================
# DESCARGA DAILY
# ======================
N_BARS_DAILY = 365 * 4 + 30

daily_data = request.get_hist(
    symbol=SYMBOL,
    exchange=EXCHANGE,
    interval=Interval.daily,
    n_bars=N_BARS_DAILY,
)

# ======================
# DESCARGA MONTHLY
# ======================
N_BARS_MONTHLY = 12 * 4 + 1

monthly_data = request.get_hist(
    symbol=SYMBOL,
    exchange=EXCHANGE,
    interval=Interval.monthly,
    n_bars=N_BARS_MONTHLY,
)

# ======================
# FUNCIÓN COMÚN DE PROCESADO
# ======================
def process_df(df_raw, frequency):
    df = df_raw.reset_index()

    df["datetime"] = pd.to_datetime(df["datetime"])
    df["year"] = df["datetime"].dt.year.astype(str)
    df["year_month_day"] = df["datetime"].dt.strftime("%Y-%m-%d")

    # Color vela
    df["candle_color"] = "doji"
    df.loc[df["close"] > df["open"], "candle_color"] = "green"
    df.loc[df["close"] < df["open"], "candle_color"] = "red"

    # Features
    df["range"] = df["high"] - df["low"]
    df["relative_volatility"] = df["range"] / df["close"]
    df["return_pct"] = ((df["close"] - df["open"]) / df["open"]) * 100

    # ======================
    # EXPORTAR POR AÑO
    # ======================
    for year in YEARS:
        df_year = df[df["year"] == year].copy()

        if df_year.empty:
            continue

        # 🔥 ahora sí eliminamos columnas
        df_year = df_year.drop(columns=["datetime", "symbol", "year"], errors="ignore")

        folder = f"{BASE_PATH}/YEAR={year}"
        os.makedirs(folder, exist_ok=True)

        df_year.to_csv(
            f"{folder}/{SYMBOL}_{frequency}.csv",
            index=False
        )

        print(f"✅ {frequency.upper()} guardado para {year}")

# ======================
# EJECUCIÓN
# ======================
process_df(daily_data, frequency="daily")
process_df(monthly_data, frequency="monthly")


✅ DAILY guardado para 2022
✅ DAILY guardado para 2023
✅ DAILY guardado para 2024
✅ DAILY guardado para 2025
✅ MONTHLY guardado para 2022
✅ MONTHLY guardado para 2023
✅ MONTHLY guardado para 2024
✅ MONTHLY guardado para 2025
